<a href="https://colab.research.google.com/github/PeteH-89/GEOG5003M_Project/blob/main/GEOG5003M_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Setting up**

**Begin by installing packages, connecting to Google Drive, and importing required packages**

In [ ]:
#uncomment to install mapclassify if required
!pip install mapclassify

#uncomment to connect to drive if required
from google.colab import drive
drive.mount('/content/drive')

# import required packages
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
import geopandas as gpd
from sklearn import cluster
from sklearn.preprocessing import scale

warnings.filterwarnings("ignore")

**Read in Data and Filter LSOAs to Only North Northamptonshire**

Using LSOA data from the Open Geography Portal (https://geoportal.statistics.gov.uk/) and Census data from https://www.nomisweb.co.uk/, read in and then use .loc to filter LSOA data to only North Northants.

In [ ]:
#read in datasets from drive
edu=pd.read_csv("/content/drive/MyDrive/GEOG5003M_Project/nnc_education.csv")
print(type(edu))
imd=pd.read_csv("/content/drive/MyDrive/GEOG5003M_Project/nnc_imd.csv")
lsoa_shp=gpd.read_file("/content/drive/MyDrive/GEOG5003M_Project/ew_lsoa.geojson")

In [ ]:
#use .head() to work out which column to filter on and then use .loc to filter lsoa data to only those within North Northamptonshire, finish by double checking everything is there using .explore
lsoa_shp.head()
nnc_lsoa=lsoa_shp.loc[lsoa_shp["LSOA21NM"].str.contains("North Northamptonshire")]
nnc_lsoa.to_file("nnc_lsoa.geojson", driver="GeoJSON")
nnc_lsoa.explore()

# **Data Exploration and Cleansing**

Check out the data we have downloaded - Look for any NAs, drop any unnecessary columns and do some basic checks on distributions etc.

In [ ]:
#start by checking dimensions of datasets, look for NAs and if present determine what to do with them (as using ONS data this is unlikely to be a problem)
print(edu.info())
print(edu.isna().sum())
print(imd.info())
print(imd.isna().sum())
print(nnc_lsoa.info())
print(nnc_lsoa.isna().sum())

All of the dataframes have 205 entries and 0 NAs, this is a good start, but the column names in the education and deprivation data need some adjustments to be usable.

In [9]:
#rename the columns in edu and imd dataframes
edu.columns=['lsoa_name', 'lsoacode', 'pop', 'pop_pct', 'noqual_raw', 'noqual_pct', 'l1_raw', 'l1_pct', 'l2_raw', 'l2_pct', 'apprenticeship', 'apprenticeship_pct', 'l3_raw', 'l3_pct', 'l4_raw', 'l4_pct', 'other_raw', 'other_pct']
imd.columns=['lsoa_name', 'lsoacode', 'households', 'pop_pct', 'nodep_raw', 'nodep_pct', '1dim_raw', '1dim_pct', '2dim_raw', '2dim_pct', '3dim_raw', '3dim_pct', '4dim_raw', '4dim_pct']
#define as new dataframes for only percentage data with lsoa_name dropped as this is effectively duplicated by LSOA Code and only one is needed to join to geojson, pop_pct columns dropped as these are 100%. Raw population may remain useful however.
edupct=edu.drop(['lsoa_name', 'pop_pct', 'noqual_raw', 'l1_raw', 'l2_raw', 'apprenticeship', 'l3_raw', 'l4_raw', 'other_raw'], axis=1)
deppct=imd.drop(['lsoa_name', 'pop_pct', 'nodep_raw', '1dim_raw', '2dim_raw', '3dim_raw', '4dim_raw'], axis=1)
#join the two percentage tables and define as new dataframe.
edu_dep=pd.DataFrame(pd.merge(left=edupct, right=deppct, how='left', on='lsoacode'))
print(edu_dep.info())
edu_dep.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   lsoacode            205 non-null    object 
 1   pop                 205 non-null    int64  
 2   noqual_pct          205 non-null    float64
 3   l1_pct              205 non-null    float64
 4   l2_pct              205 non-null    float64
 5   apprenticeship_pct  205 non-null    float64
 6   l3_pct              205 non-null    float64
 7   l4_pct              205 non-null    float64
 8   other_pct           205 non-null    float64
 9   households          205 non-null    int64  
 10  nodep_pct           205 non-null    float64
 11  1dim_pct            205 non-null    float64
 12  2dim_pct            205 non-null    float64
 13  3dim_pct            205 non-null    float64
 14  4dim_pct            205 non-null    float64
dtypes: float64(12), int64(2), object(1)
memory usage: 24.2+ K

,pop,noqual_pct,l1_pct,l2_pct,apprenticeship_pct,l3_pct,l4_pct,other_pct,households,nodep_pct,1dim_pct,2dim_pct,3dim_pct,4dim_pct
count,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000
mean,1410.595122,20.256585,12.032195,15.368780,6.320488,16.674634,26.256098,3.090732,726.487805,48.598049,34.137073,13.763902,3.334634,0.164390
std,297.189527,6.042474,2.088422,1.761712,1.128321,2.508992,7.113190,0.909246,155.050422,9.321065,3.371302,4.660836,2.379141,0.233159
min,875.000000,7.300000,6.500000,9.500000,2.500000,10.100000,13.900000,1.300000,451.000000,27.300000,21.700000,2.800000,0.000000,0.000000
25%,1194.000000,15.700000,10.500000,14.300000,5.700000,15.000000,20.700000,2.500000,617.000000,41.300000,32.000000,9.900000,1.400000,0.000000
50%,1351.000000,20.200000,12.200000,15.500000,6.300000,16.800000,25.300000,3.000000,684.000000,49.100000,34.500000,13.600000,2.500000,0.100000
75%,1613.000000,24.400000,13.400000,16.400000,6.900000,18.200000,31.200000,3.600000,830.000000,56.100000,36.400000,17.100000,5.100000,0.300000
max,2126.000000,35.000000,18.300000,25.700000,9.500000,26.300000,46.200000,6.700000,1146.000000,75.400000,41.200000,25.100000,11.700000,1.300000


The data description appears to be fairly reasonable - Our area has a blend of rural areas and small and mid sized towns and that would reasonably account for the variation between LSOAs. One notable thing is that there are certain LSOAs without any severe (3 or 4 dimensions) deprivation.